In [5]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Load data
df = pd.read_csv("deliveries.csv")

# Basic Filtering
df1 = df[["match_id", "batting_team", "bowling_team", "over", "ball", "total_runs", "is_wicket"]]
df1["total_runs"] = pd.to_numeric(df1["total_runs"], errors="coerce")
df1["is_wicket"] = df1["is_wicket"].apply(lambda x: 1 if x != 0 else 0)

# Feature Engineering
df1["cum_runs"] = df1.groupby("match_id")["total_runs"].cumsum()
df1["balls_bowled"] = (df1["over"] - 1) * 6 + df1["ball"]
df1["balls_left"] = 120 - df1["balls_bowled"]
df1["balls_left"] = df1["balls_left"].clip(lower=0)

df1["wickets_fallen"] = df1.groupby("match_id")["is_wicket"].cumsum()
df1["wickets_left"] = 10 - df1["wickets_fallen"]

# CRR (Run Rate)
df1["crr"] = (df1["cum_runs"] * 6) / df1["balls_bowled"]

# Last 5 overs runs
df1["last5_runs"] = (
    df1.groupby("match_id")["total_runs"]
    .rolling(30).sum()
    .reset_index(level=0, drop=True)
)

# Phase
df1["phase"] = df1["balls_bowled"].apply(lambda x: 
    "Powerplay" if x <= 36 else "Middle" if x <= 90 else "Death"
)

# Get Final Score (Target)
inning_runs = df1.groupby(["match_id", "batting_team"])["total_runs"].sum().reset_index().rename(columns={"total_runs": "target"})
df1 = df1.merge(inning_runs, on=["match_id", "batting_team"])

# Clean Dataset
df1 = df1[(df1["target"] > 50) & (df1["balls_bowled"] > 30)] # Remove early noise
df1.dropna(inplace=True)

# Select FINAL features (No RRR, No Pressure)
final_df = df1[[
    "batting_team", "bowling_team", "phase", "cum_runs", 
    "balls_left", "wickets_left", "crr", "last5_runs", "target"
]]

X = final_df.drop(columns=["target"])
y = final_df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Pipeline
trf = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown='ignore'), ["batting_team", "bowling_team", "phase"])
], remainder="passthrough")

pipe = Pipeline(steps=[
    ("step1", trf),
    ("step2", StandardScaler(with_mean=False)),
    ("step3", LGBMRegressor(n_estimators=1000, learning_rate=0.02, max_depth=10, random_state=42))
])

pipe.fit(X_train, y_train)

# Save
pickle.dump(pipe, open("beg.pkl", "wb"))
print("✅ Optimized Model Saved!")

C:\Users\mirza\AppData\Local\Temp\ipykernel_12748\576393442.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["total_runs"] = pd.to_numeric(df1["total_runs"], errors="coerce")
C:\Users\mirza\AppData\Local\Temp\ipykernel_12748\576393442.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["is_wicket"] = df1["is_wicket"].apply(lambda x: 1 if x != 0 else 0)
C:\Users\mirza\AppData\Local\Temp\ipykernel_12748\576393442.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040891 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 790
[LightGBM] [Info] Number of data points in the train set: 143525, number of used features: 43
[LightGBM] [Info] Start training from score 161.628288
✅ Optimized Model Saved!
